# Chapter 07 Companion Notebook: Logit: Customer Churn

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch07_Logit_Customer_Churn.ipynb)

This notebook accompanies Chapter 07 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).



# Customer Churn Logistic Regression Analysis

### Use “Customer churn.csv”
The company implements marketing campaigns specifically designed to retain customers at risk of discontinuing their services or products. These campaigns aim to reduce customer churn by addressing the reasons for dissatisfaction or disengagement. The effectiveness of these efforts is expected to be moderate, with a predicted **success rate of 30%**, meaning that half of the targeted customers are likely to respond positively to the campaigns and continue their relationship with the company.
- Churn: 1 if the customer has left the company or discontinued their service
- AccountWeeks: the number of weeks a customer has been with the company
- DataPlan: 1 if the customer is subscribed to a data plan
- DataUsage: the amount of data a customer has consumed
- CustServCalls: the number of times a customer has contacted customer service
- DayMins: the total number of minutes a customer has used during the day
- DayCalls: the number of calls a customer has made during the day
- MonthlyCharge: the amount a customer is billed each month
- OverageFee: charges a customer incurs for exceeding their plan's limits
- RoamMins: the number of minutes a customer spends on roaming calls

### 1. Divide the data into 75% training and 25% test set (use random_state=1). Run the following logistic regression model on the training data and report coefficients.
- Dependent varirable: Churn
- Independent varirable: 'AccountWeeks', 'ContractRenewal', 'DataPlan', 'DataUsage', 'CustServCalls', 'DayMins', 'DayCalls',  'OverageFee', 'RoamMins'

In [ ]:
# Load the dataset
import pandas as pd
df = pd.read_csv("Customer churn.csv")
df.head()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

x = df[['AccountWeeks', 'DataPlan', 'DataUsage', 'CustServCalls', 'DayMins',
        'DayCalls',  'OverageFee', 'RoamMins']]
y = df['Churn']

# Splitting the dataset into training and testing sets
xtrain, xtest, ytrain, ytest = train_test_split(x, y, random_state=1)

In [ ]:
# Fit the logistic regression model

m = LogisticRegression(max_iter=10000)
m.fit(xtrain, ytrain)

# Display the coefficients

coef = pd.DataFrame({'Variable': x.columns, 'Coefficient': m.coef_[0]})
print("Logistic regression coefficients:\n", coef)

### 2. Interpret the statistically significant coefficients at a 5% significance level.

In [ ]:
import numpy as np
from sklearn.utils import resample
import statsmodels.api as sm

# Adding constant to the training data
xtrain_con = sm.add_constant(xtrain)

# Initializing a logistic regression model
m1 = sm.Logit(ytrain, xtrain_con)

# Fitting the logistic regression model
result = m1.fit()

# Displaying the summary of the logistic regression model results.
print(result.summary())

The statistically significant coefficients at a 5% significance level (p-value < 0.05) are:
- CustServCalls: An increase in customer service calls is associated with a higher likelihood of churn.
- DayMins: More daytime minutes are positively associated with a higher likelihood of churn.
- OverageFee: Higher overage fees increase the likelihood of churn.
- RoamMins: More roaming minutes are associated with a higher likelihood of churn.

### 3. What is the percentage of correct classifications on the test dataset?

In [ ]:
# Predict on the test set

ypred = m.predict(xtest)
accuracy = accuracy_score(ytest, ypred)
print("Accuracy of the model:", accuracy)

### 4. Predict the response probabilities on the test dataset. Save the probabilities as a dataframe.

In [ ]:
# Predict probabilities

prob = pd.DataFrame(m.predict_proba(x))
prob = prob.rename(columns={0:'pred_zero', 1:'pred_one'})
prob.head()

### 5. Random sampling: How many customers can you expect to retain if a random 10% of the customers in the test dataset are targeted in the marketing campaign?

In [ ]:
print(len(y))
print(len(ytest))
sum(y)  # sum of ones in the whole data (=churn)

In [ ]:
# % of churn in the test data

churn_pct=sum(ytest)/len(ytest)
churn_pct

In [ ]:
# We expect the Same % of people to be retained in random sampling with 30% success (retention) rate

sample_size = 0.1*len(ytest)
0.3*sample_size*churn_pct

### 7. Data science approach: How many customers can you expect to retain if you target the top 10% of customers with the highest predicted probabilities in the test dataset in the marketing campaign?

In [ ]:
xtest  # index of text data

In [ ]:
prob_test = prob.loc[xtest.index]
df1=pd.concat([ytest, xtest, prob_test], axis=1)

In [ ]:
# Sort by descending order of prob

df1=df1.sort_values(by='pred_one', ascending=False)

In [ ]:
# The number of churn=1 in top 10% with 30% success (retention) rate

0.3*sum(df1[0:int(sample_size)].Churn)

- int(): Converts to an integer, discarding any decimal part.
- sum(): Computes the sum of the churn values in the first 10% of rows.

### 8. The marketing campaign costs \\$5 per customer, and the company earns $50 in revenue for each customer retained as a result of the campaign. Compare the profits from the campaign under the "random sampling" and the "data science approach". Assume there are no additional costs.

In [ ]:
# Cost and revenue parameters

cost_per_cust = 5
rev_per_cust = 50
retention = 0.3

# Random targeting of 10% of customers

rand_target = int(0.1 * len(xtest))
rand_retained = rand_target * retention * (sum(ytest) / len(ytest))
rand_rev = rand_retained * rev_per_cust
rand_cost = rand_target * cost_per_cust
rand_profit = rand_rev - rand_cost

# Smart scenario: Target top 10% of customers based on predicted probabilities

ds_retained = sum(df1[0:int(0.1 * len(xtest))].Churn) * retention
ds_rev = ds_retained * rev_per_cust
ds_cost = int(0.1 * len(xtest)) * cost_per_cust
ds_profit = ds_rev - ds_cost

# Display results

print(f"Random targeting profit: ${rand_profit:.2f}")
print(f"Data science profit: ${ds_profit:.2f}")